In [ ]:
# --------------------------------
# Step 1 — Load full-corpus KWIC data
# --------------------------------

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "outputs"

kwic_path = OUTPUT_DIR / "institution_kwic_results_v2.csv"

kwic = pd.read_csv(kwic_path)

print("KWIC data loaded successfully.")
print("Shape:", kwic.shape)

print("\nColumns:")
print(kwic.columns.tolist())

print("\nCategory counts:")
print(
    kwic["category"]
    .value_counts(dropna=False)
)

print("\nUnique transcripts by category:")
print(
    kwic.groupby("category")["transcript_id"]
    .nunique()
    .sort_values(ascending=False)
)

In [ ]:
# --------------------------------
# Step 2 — Diagnose duplicate and overlapping KWIC passages
# --------------------------------

ANALYSIS_CATEGORIES = [
    "police",
    "legal",
    "health",
    "welfare_state",
    "support_sector"
]

kwic_5 = kwic[
    kwic["category"].isin(ANALYSIS_CATEGORIES)
].copy()

# Exact duplicate contexts
exact_duplicate_mask = kwic_5.duplicated(
    subset=["transcript_id", "category", "context"],
    keep=False
)

exact_duplicates = kwic_5[exact_duplicate_mask].copy()

print("Total KWIC rows:", len(kwic_5))
print(
    "Rows involved in exact duplicate contexts:",
    len(exact_duplicates)
)

print(
    "Unique exact duplicate transcript-category-context combinations:",
    exact_duplicates[
        ["transcript_id", "category", "context"]
    ].drop_duplicates().shape[0]
)

print("\nExact duplicate rows by category:")
print(
    exact_duplicates["category"]
    .value_counts()
)

In [ ]:
# --------------------------------
# Step 3 — Diagnose nearby KWIC hits
# --------------------------------

# Sort hits by transcript, category, and position
kwic_sorted = (
    kwic_5
    .sort_values(
        ["transcript_id", "category", "match_start"]
    )
    .copy()
)

# Previous hit information within the same transcript-category group
kwic_sorted["prev_match_start"] = (
    kwic_sorted
    .groupby(["transcript_id", "category"])["match_start"]
    .shift(1)
)

kwic_sorted["prev_match_end"] = (
    kwic_sorted
    .groupby(["transcript_id", "category"])["match_end"]
    .shift(1)
)

kwic_sorted["prev_keyword"] = (
    kwic_sorted
    .groupby(["transcript_id", "category"])["keyword"]
    .shift(1)
)

# Distance between current hit and previous hit
kwic_sorted["gap_from_previous"] = (
    kwic_sorted["match_start"]
    - kwic_sorted["prev_match_end"]
)

# Summarise how many hits are very close together
thresholds = [0, 50, 100, 200, 300, 500]

print("Nearby-hit diagnostics:")
print("-----------------------")

for threshold in thresholds:
    n = (
        kwic_sorted["gap_from_previous"]
        .between(0, threshold, inclusive="both")
        .sum()
    )
    
    pct = n / len(kwic_sorted) * 100
    
    print(
        f"Gap <= {threshold:>3} characters: "
        f"{n:>5} hits ({pct:.2f}%)"
    )

# Cases where match positions themselves overlap
overlapping_hits = kwic_sorted[
    kwic_sorted["gap_from_previous"] < 0
].copy()

print("\nDirectly overlapping match positions:")
print(len(overlapping_hits))

print("\nGap distribution:")
print(
    kwic_sorted["gap_from_previous"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

In [ ]:
# --------------------------------
# Step 4 — Estimate KWIC window spans
# --------------------------------

kwic_windows = kwic_5.copy()

# Make sure text columns contain strings
for col in ["left_context", "matched_text", "right_context"]:
    kwic_windows[col] = kwic_windows[col].fillna("").astype(str)

# Estimate the start/end position of each complete KWIC window
kwic_windows["window_start"] = (
    kwic_windows["match_start"]
    - kwic_windows["left_context"].str.len()
)

kwic_windows["window_end"] = (
    kwic_windows["match_end"]
    + kwic_windows["right_context"].str.len()
)

# Prevent negative starts
kwic_windows["window_start"] = (
    kwic_windows["window_start"]
    .clip(lower=0)
)

# Sort before overlap diagnosis
kwic_windows = (
    kwic_windows
    .sort_values(
        ["transcript_id", "category", "window_start", "window_end"]
    )
    .reset_index(drop=True)
)

# Previous window end within each transcript-category
kwic_windows["prev_window_end"] = (
    kwic_windows
    .groupby(["transcript_id", "category"])["window_end"]
    .shift(1)
)

# Does this KWIC context overlap with the preceding context?
kwic_windows["overlaps_previous_window"] = (
    kwic_windows["window_start"]
    <= kwic_windows["prev_window_end"]
)

print("Total KWIC windows:", len(kwic_windows))

print(
    "Windows overlapping previous KWIC window:",
    kwic_windows["overlaps_previous_window"].sum()
)

print(
    "Percentage:",
    f"{kwic_windows['overlaps_previous_window'].mean() * 100:.2f}%"
)

print("\nOverlapping windows by category:")
print(
    kwic_windows.loc[
        kwic_windows["overlaps_previous_window"],
        "category"
    ].value_counts()
)

In [ ]:
# --------------------------------
# Step 5 — Merge overlapping KWIC windows
# --------------------------------

merged_rows = []

for (transcript_id, category), group in (
    kwic_windows
    .groupby(["transcript_id", "category"])
):

    group = group.sort_values(
        ["window_start", "window_end"]
    )

    current_start = None
    current_end = None
    current_contexts = []
    current_keywords = []

    for _, row in group.iterrows():

        start = row["window_start"]
        end = row["window_end"]

        if current_start is None:
            # start first merged passage
            current_start = start
            current_end = end
            current_contexts = [row["context"]]
            current_keywords = [row["keyword"]]

        elif start <= current_end:
            # overlapping window -> merge
            current_end = max(current_end, end)
            current_contexts.append(row["context"])
            current_keywords.append(row["keyword"])

        else:
            # save completed merged passage
            merged_rows.append({
                "transcript_id": transcript_id,
                "category": category,
                "merged_start": current_start,
                "merged_end": current_end,
                "n_windows": len(current_contexts),
                "keywords": sorted(set(current_keywords)),
                "contexts": current_contexts
            })

            # start new passage
            current_start = start
            current_end = end
            current_contexts = [row["context"]]
            current_keywords = [row["keyword"]]

    # save final passage in group
    if current_start is not None:
        merged_rows.append({
            "transcript_id": transcript_id,
            "category": category,
            "merged_start": current_start,
            "merged_end": current_end,
            "n_windows": len(current_contexts),
            "keywords": sorted(set(current_keywords)),
            "contexts": current_contexts
        })


merged_passages = pd.DataFrame(merged_rows)

print("Raw KWIC windows:", len(kwic_windows))
print("Merged contextual passages:", len(merged_passages))
print(
    "Reduction:",
    len(kwic_windows) - len(merged_passages)
)

print("\nMerged passages by category:")
print(
    merged_passages["category"]
    .value_counts()
)

print("\nNumber of original windows per merged passage:")
print(
    merged_passages["n_windows"]
    .describe()
)

In [ ]:
# --------------------------------
# Step 6 — Reconstruct merged passage text without overlap
# --------------------------------

def reconstruct_merged_passages(kwic_windows):
    rows = []

    for (transcript_id, category), group in (
        kwic_windows
        .groupby(["transcript_id", "category"])
    ):
        group = (
            group
            .sort_values(["window_start", "window_end"])
            .copy()
        )

        current_start = None
        current_end = None
        current_text = ""
        current_keywords = []
        n_windows = 0

        for _, row in group.iterrows():

            start = int(row["window_start"])
            end = int(row["window_end"])

            # Use the complete KWIC context as the window text
            text = str(row["context"])

            if current_start is None:
                # Start first passage
                current_start = start
                current_end = end
                current_text = text
                current_keywords = [row["keyword"]]
                n_windows = 1

            elif start <= current_end:
                # Overlapping window:
                # append only the part extending beyond current passage
                overlap_chars = current_end - start

                if end > current_end:
                    new_chars = end - current_end

                    # Take only the non-overlapping suffix
                    if new_chars > 0:
                        current_text += text[-new_chars:]

                    current_end = end

                current_keywords.append(row["keyword"])
                n_windows += 1

            else:
                # Save completed passage
                rows.append({
                    "transcript_id": transcript_id,
                    "category": category,
                    "passage_start": current_start,
                    "passage_end": current_end,
                    "n_windows": n_windows,
                    "keywords": sorted(set(current_keywords)),
                    "passage_text": current_text
                })

                # Start next passage
                current_start = start
                current_end = end
                current_text = text
                current_keywords = [row["keyword"]]
                n_windows = 1

        # Save final passage
        if current_start is not None:
            rows.append({
                "transcript_id": transcript_id,
                "category": category,
                "passage_start": current_start,
                "passage_end": current_end,
                "n_windows": n_windows,
                "keywords": sorted(set(current_keywords)),
                "passage_text": current_text
            })

    return pd.DataFrame(rows)


passages = reconstruct_merged_passages(kwic_windows)

print("Reconstructed passages:", len(passages))

print("\nPassages by category:")
print(passages["category"].value_counts())

print("\nPassage length distribution:")
print(passages["passage_text"].str.len().describe())

display(
    passages[
        [
            "transcript_id",
            "category",
            "n_windows",
            "keywords",
            "passage_text"
        ]
    ].head(10)
)

In [ ]:
# --------------------------------
# Step 7 — Build transcript-category analytical documents
# --------------------------------

tfidf_docs = (
    passages
    .groupby(["transcript_id", "category"], as_index=False)
    .agg(
        text=("passage_text", " ".join),
        n_passages=("passage_text", "size"),
        n_kwic_windows=("n_windows", "sum")
    )
)

tfidf_docs["char_count"] = tfidf_docs["text"].str.len()

print("Analytical documents:", len(tfidf_docs))

print("\nDocuments by category:")
print(
    tfidf_docs["category"]
    .value_counts()
    .sort_index()
)

print("\nNumber of passages per document:")
print(tfidf_docs["n_passages"].describe())

print("\nDocument character length:")
print(tfidf_docs["char_count"].describe())

display(tfidf_docs.head(10))

In [ ]:
from importlib.metadata import version

print("click:", version("click"))
print("typer:", version("typer"))
print("spacy:", version("spacy"))

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
# --------------------------------
# Step 8A — Load spaCy model and test lemmatisation
# --------------------------------

import spacy

nlp = spacy.load("en_core_web_sm")

test_text = """
The police came to the house.
She reported the incident and later phoned the station.
Support workers were helping women and children.
"""

doc = nlp(test_text)

test_results = [
    (token.text, token.pos_, token.lemma_)
    for token in doc
    if token.is_alpha
]

for item in test_results:
    print(item)

In [ ]:
# --------------------------------
# Step 8B — Define and test linguistic preprocessing
# --------------------------------

# POS categories retained for TF-IDF analysis
KEEP_POS = {"NOUN", "VERB", "ADJ", "ADV"}

def preprocess_text(text):
    """
    Preprocess text for TF-IDF:
    - lowercase through lemma form
    - lemmatise words
    - remove stopwords
    - remove punctuation, numbers and non-alphabetic tokens
    - retain nouns, verbs, adjectives and adverbs
    """
    
    doc = nlp(str(text))
    
    tokens = []
    
    for token in doc:
        # Keep alphabetic content words only
        if not token.is_alpha:
            continue
        
        # Remove standard English stopwords
        if token.is_stop:
            continue
        
        # Retain selected lexical POS categories
        if token.pos_ not in KEEP_POS:
            continue
        
        lemma = token.lemma_.lower().strip()
        
        # Remove empty or very short residual tokens
        if len(lemma) < 2:
            continue
        
        tokens.append(lemma)
    
    return " ".join(tokens)


# Test preprocessing
processed_test = preprocess_text(test_text)

print("Original:")
print(test_text)

print("\nProcessed:")
print(processed_test)

In [ ]:
# --------------------------------
# Step 8C — Apply preprocessing to analytical documents
# --------------------------------

from tqdm.auto import tqdm

tqdm.pandas()

tfidf_docs["processed_text"] = (
    tfidf_docs["text"]
    .fillna("")
    .progress_apply(preprocess_text)
)

tfidf_docs["processed_word_count"] = (
    tfidf_docs["processed_text"]
    .str.split()
    .str.len()
)

print("Documents processed:", len(tfidf_docs))

print("\nProcessed word-count distribution:")
print(
    tfidf_docs["processed_word_count"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
)

print("\nEmpty documents after preprocessing:")
print(
    (tfidf_docs["processed_word_count"] == 0).sum()
)

display(
    tfidf_docs[
        [
            "transcript_id",
            "category",
            "n_passages",
            "processed_word_count",
            "processed_text"
        ]
    ].head(10)
)

In [ ]:
# --------------------------------
# Step 8D — Inspect frequent processed terms
# --------------------------------

from collections import Counter

all_tokens = " ".join(
    tfidf_docs["processed_text"].fillna("")
).split()

term_counts = Counter(all_tokens)

top_terms = pd.DataFrame(
    term_counts.most_common(50),
    columns=["term", "count"]
)

print("Total processed tokens:", len(all_tokens))
print("Unique processed terms:", len(term_counts))

display(top_terms)

In [ ]:
# --------------------------------
# Step 8E — Corpus-specific stopwords
# --------------------------------

CUSTOM_STOPWORDS = {
    # transcript / speaker labels
    "interviewer",
    "respondent",

    # high-frequency conversational / discourse terms
    "know",
    "yeah",
    "yes",
    "okay",
    "ok",
    "actually",
    "kind",
    "sort",
    "mean",
    "thing",
    "lot"
}

print("Custom stopwords:")
print(sorted(CUSTOM_STOPWORDS))
print("\nNumber of custom stopwords:", len(CUSTOM_STOPWORDS))

In [ ]:
# --------------------------------
# Step 8F — Apply corpus-specific stopwords
# --------------------------------

def remove_custom_stopwords(text):
    tokens = text.split()
    tokens = [
        token for token in tokens
        if token not in CUSTOM_STOPWORDS
    ]
    return " ".join(tokens)

tfidf_docs["tfidf_text"] = (
    tfidf_docs["processed_text"]
    .fillna("")
    .apply(remove_custom_stopwords)
)

# Diagnostics
tfidf_docs["tfidf_word_count"] = (
    tfidf_docs["tfidf_text"]
    .str.split()
    .str.len()
)

print("Documents:", len(tfidf_docs))

print("\nTF-IDF input word-count distribution:")
print(
    tfidf_docs["tfidf_word_count"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
)

print(
    "\nEmpty documents after custom stopword removal:",
    (tfidf_docs["tfidf_text"].str.strip() == "").sum()
)

display(
    tfidf_docs[
        [
            "transcript_id",
            "category",
            "processed_word_count",
            "tfidf_word_count",
            "tfidf_text"
        ]
    ].head(10)
)

In [ ]:
# --------------------------------
# Step 8G — Final vocabulary diagnostic
# --------------------------------

from collections import Counter

final_tokens = []

for text in tfidf_docs["tfidf_text"]:
    final_tokens.extend(text.split())

final_counts = Counter(final_tokens)

final_freq = (
    pd.DataFrame(
        final_counts.most_common(50),
        columns=["term", "count"]
    )
)

print("Total TF-IDF input tokens:", len(final_tokens))
print("Unique TF-IDF input terms:", len(final_counts))

display(final_freq)

In [ ]:
CUSTOM_STOPWORDS.add("inaudible")

tfidf_docs["tfidf_text"] = (
    tfidf_docs["processed_text"]
    .fillna("")
    .apply(remove_custom_stopwords)
)

tfidf_docs["tfidf_word_count"] = (
    tfidf_docs["tfidf_text"]
    .str.split()
    .str.len()
)

print("Custom stopwords:", sorted(CUSTOM_STOPWORDS))
print("Documents:", len(tfidf_docs))
print("Empty documents:", (tfidf_docs["tfidf_word_count"] == 0).sum())

In [ ]:
# --------------------------------
# Step 9A — Build baseline TF-IDF matrix
# --------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer

baseline_vectorizer = TfidfVectorizer(
    lowercase=False,      # already lowercased during preprocessing
    ngram_range=(1, 1),   # unigram baseline
    min_df=3,             # term must appear in at least 3 documents
    max_df=0.90,          # remove terms appearing in >90% of documents
    sublinear_tf=True
)

baseline_tfidf_matrix = baseline_vectorizer.fit_transform(
    tfidf_docs["tfidf_text"]
)

baseline_feature_names = baseline_vectorizer.get_feature_names_out()

print("Baseline TF-IDF matrix shape:", baseline_tfidf_matrix.shape)
print("Documents:", baseline_tfidf_matrix.shape[0])
print("Features:", baseline_tfidf_matrix.shape[1])

print("\nFirst 30 features:")
print(baseline_feature_names[:30])

In [ ]:
# --------------------------------
# Step 9B — Inspect baseline TF-IDF terms
# --------------------------------

import numpy as np
import pandas as pd

# Mean TF-IDF score of each term across all 627 documents
mean_tfidf = np.asarray(
    baseline_tfidf_matrix.mean(axis=0)
).ravel()

baseline_top_terms = (
    pd.DataFrame({
        "term": baseline_feature_names,
        "mean_tfidf": mean_tfidf
    })
    .sort_values("mean_tfidf", ascending=False)
    .reset_index(drop=True)
)

print("Top 50 terms by mean TF-IDF:")
display(baseline_top_terms.head(50))

In [ ]:
# --------------------------------
# Step 9C — Top TF-IDF terms by institution category
# --------------------------------

import numpy as np
import pandas as pd

categories = tfidf_docs["category"].unique()

category_top_terms = {}

for category in categories:
    
    # Find documents belonging to this category
    mask = tfidf_docs["category"].values == category
    
    # Mean TF-IDF score within this category
    category_mean = np.asarray(
        baseline_tfidf_matrix[mask].mean(axis=0)
    ).ravel()
    
    # Create ranked table
    top_terms = (
        pd.DataFrame({
            "term": baseline_feature_names,
            "mean_tfidf": category_mean
        })
        .sort_values("mean_tfidf", ascending=False)
        .reset_index(drop=True)
    )
    
    category_top_terms[category] = top_terms
    
    print("\n" + "=" * 60)
    print(f"Category: {category}")
    print(f"Documents: {mask.sum()}")
    print("=" * 60)
    
    display(top_terms.head(30))

In [ ]:
# --------------------------------
# Step 9D — Cross-category TF-IDF distinctiveness
# --------------------------------

import numpy as np
import pandas as pd

categories = sorted(tfidf_docs["category"].unique())

# Mean TF-IDF vector for each category
category_mean_vectors = {}

for category in categories:
    mask = tfidf_docs["category"].values == category
    
    category_mean_vectors[category] = np.asarray(
        baseline_tfidf_matrix[mask].mean(axis=0)
    ).ravel()

# Build cross-category mean TF-IDF table
cross_category_tfidf = pd.DataFrame(
    category_mean_vectors,
    index=baseline_feature_names
)

print("Cross-category TF-IDF matrix:")
print(cross_category_tfidf.shape)

display(cross_category_tfidf.head(20))

In [ ]:
# Step 9E — Calculate category-distinctive TF-IDF terms

import pandas as pd

distinctive_results = {}

for category in cross_category_tfidf.columns:

    # Mean TF-IDF in the target category
    target_score = cross_category_tfidf[category]

    # Mean TF-IDF across all other categories
    other_categories = [
        c for c in cross_category_tfidf.columns
        if c != category
    ]

    other_mean = cross_category_tfidf[other_categories].mean(axis=1)

    # Distinctiveness = target category - mean of other categories
    distinctive_score = target_score - other_mean

    result = pd.DataFrame({
        "term": cross_category_tfidf.index,
        "category_mean_tfidf": target_score.values,
        "other_categories_mean": other_mean.values,
        "distinctive_score": distinctive_score.values
    })

    # Highest positive difference = most distinctive for this category
    result = (
        result
        .sort_values("distinctive_score", ascending=False)
        .reset_index(drop=True)
    )

    distinctive_results[category] = result

    print("=" * 60)
    print(f"Category: {category}")
    print("=" * 60)

    display(result.head(30))

### Step 9E — Category-distinctive TF-IDF

Cross-category comparison shows clear lexical differences between the five institution categories.

- **Health:** mental health, counselling, doctors, hospitals, disability and psychological conditions.
- **Legal:** courts, solicitors, judges, legal procedures, evidence and prosecution.
- **Police:** reporting, arrest, officers, investigation and police contact.
- **Support sector:** refuges, charities, helplines, support workers and voluntary organisations.
- **Welfare/state:** housing, income, rent, council services, benefits and financial issues.

Overall, the distinctive TF-IDF results suggest that the five categories capture meaningfully different institutional contexts. Some generic or potentially noisy terms remain, so TF-IDF is treated as an exploratory comparison rather than direct evidence of survivors' evaluations of institutions.

In [ ]:
# Step 9G — Create summary table of top distinctive TF-IDF terms

TOP_N = 10

summary_rows = []

for category, df in distinctive_results.items():
    top_terms = df.head(TOP_N)

    summary_rows.append({
        "category": category,
        "top_distinctive_terms": ", ".join(top_terms["term"].tolist())
    })

tfidf_summary = pd.DataFrame(summary_rows)

display(tfidf_summary)

In [ ]:
# Step 9H — Treemap of top distinctive TF-IDF terms by category

import matplotlib.pyplot as plt
import squarify

TOP_N = 10

fig, axes = plt.subplots(
    3, 2,
    figsize=(16, 15)
)

axes = axes.flatten()

for i, category in enumerate(ANALYSIS_CATEGORIES):
    
    ax = axes[i]

    # Get top distinctive terms for this category
    df = distinctive_results[category].head(TOP_N).copy()

    labels = [
        f"{term}\n{score:.3f}"
        for term, score in zip(
            df["term"],
            df["distinctive_score"]
        )
    ]

    sizes = df["distinctive_score"].clip(lower=0)

    squarify.plot(
        sizes=sizes,
        label=labels,
        alpha=0.8,
        ax=ax
    )

    ax.set_title(
        category.replace("_", " ").title(),
        fontsize=15,
        fontweight="bold"
    )

    ax.axis("off")


# Remove unused sixth panel
for j in range(len(ANALYSIS_CATEGORIES), len(axes)):
    fig.delaxes(axes[j])


fig.suptitle(
    "Top Distinctive TF-IDF Terms by Institution Category",
    fontsize=19,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()

plt.show()

In [ ]:
# Step 9I — Save final TF-IDF outputs

# 1. Save full cross-category TF-IDF matrix
cross_category_tfidf.to_csv(
    OUTPUT_DIR / "step9_cross_category_tfidf_matrix.csv"
)

# 2. Save distinctive terms for each category
for category, df in distinctive_results.items():
    df.to_csv(
        OUTPUT_DIR / f"step9_distinctive_terms_{category}.csv",
        index=False
    )

# 3. Save compact summary table
tfidf_summary.to_csv(
    OUTPUT_DIR / "step9_tfidf_summary_top10.csv",
    index=False
)

print("Step 9I outputs saved:")
print("- Cross-category TF-IDF matrix")
print("- Distinctive-term tables for 5 categories")
print("- Top-10 summary table")
print(f"\nOutput directory: {OUTPUT_DIR}")

In [ ]:
# Export document-level dataset for downstream NMF analysis

tfidf_docs.to_csv(
    OUTPUT_DIR / "step9_tfidf_documents.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "step9_tfidf_documents.csv")
print("Shape:", tfidf_docs.shape)
print(tfidf_docs.columns.tolist())

In [ ]:
# Export passage-level data for NMF artefact diagnostics

passages.to_csv(
    OUTPUT_DIR / "step9_passages_for_nmf.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "step9_passages_for_nmf.csv")
print("Shape:", passages.shape)
print(passages.columns.tolist())